# Huấn Luyện & Đánh Giá Mô Hình CKAN Trên GPU (Google Colab)
### DSS Movie Recommendation System: Matrix Factorization (Baseline) vs CKAN (Knowledge Graph)

Notebook này phục vụ cho đồ án **Hệ Thống Đề Xuất Phim Dựa Trên Đồ Thị Tri Thức (CKAN)** với quy trình thực nghiệm độc lập:
1. **Huấn luyện Mô hình Cơ sở (Baseline)**: Matrix Factorization (MF) thuần túy không dùng Đồ thị Tri thức.
2. **Huấn luyện Mô hình Đề xuất**: CKAN (Collaborative Knowledge-aware Attentive Network) tận dụng tri thức đa tầng (Director, Actor, Genre).
3. **Nghiên cứu thực nghiệm (Ablation Study)**: Đặt 2 mô hình lên cùng bàn cân để so sánh trực quan độ chính xác ROC-AUC, F1-Score.
4. **Xuất mô hình**: Tải file checkpoint `ckan_model.pt` tương thích 100% về máy tính để tích hợp ngay vào Backend FastAPI của hệ thống.

> **Lưu ý Colab**: Hãy đảm bảo bạn đã bật GPU: **Runtime (Thời gian chạy) → Change runtime type (Thay đổi loại phần cứng) → Chọn T4 GPU**.


In [ ]:
# ============================================================
# 1. KIỂM TRA MÔI TRƯỜNG & GPU
# ============================================================
import torch

print("=== KIỂM TRA PHẦN CỨNG ===")
print("Phiên bản PyTorch :", torch.__version__)
print("CUDA khả dụng     :", torch.cuda.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Tên GPU           :", torch.cuda.get_device_name(0))
    print(f"Bộ nhớ GPU        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("[Cảnh báo] Đang dùng CPU. Khuyên bạn nên đổi sang GPU T4 trong Runtime -> Change runtime type.")



In [ ]:
# ============================================================
# 2. CÀI ĐẶT THƯ VIỆN BỔ TRỢ
# ============================================================
!pip install -q --upgrade scikit-learn matplotlib seaborn pandas tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, f1_score
from collections import defaultdict
import os
import shutil
import urllib.request
import zipfile
from tqdm import tqdm

print("[OK] Đã nạp thành công các thư viện cần thiết!")



In [ ]:
# ============================================================
# 3. THIẾT LẬP SIÊU THAM SỐ (HYPERPARAMETERS)
# ============================================================
DATASET = "movie"   # Mặc định bộ phim (MovieLens + KG Satori)

# Siêu tham số chuẩn theo paper CKAN
DIM         = 64       # Chiều vector biểu diễn Embedding (đồng nhất cho cả MF và CKAN)
N_LAYER     = 1        # Số bước lan truyền tri thức (L=1 tối ưu cho MovieLens)
UTSS        = 32       # Kích thước tập bộ ba user (User Triple Set Size)
ITSS        = 64       # Kích thước tập bộ ba item (Item Triple Set Size)
AGG         = "concat" # Cơ chế gộp embedding: 'concat' | 'sum' | 'pool'
BATCH_SIZE  = 2048     # Kích thước mini-batch huấn luyện
N_EPOCH     = 20       # Số epoch huấn luyện cho mỗi mô hình
LEARNING_RATE = 0.002  # Tốc độ học của Adam optimizer
L2_WEIGHT   = 1e-5     # Hệ số điều chuẩn L2 Weight Decay

DATA_DIR = f"./data/{DATASET}"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)

print(f"Dataset       : {DATASET}")
print(f"Embedding Dim : {DIM}")
print(f"KG Layers     : {N_LAYER}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Epochs        : {N_EPOCH}")



## 4. Tải Dữ Liệu Huấn Luyện (Dataset)
Tải trực tiếp các file ma trận đã chuẩn hóa (`ratings_final.npy`, `kg_final.npy`, `item_index2entity_id.txt`, `kg.txt`) từ kho GitHub dự án.


In [ ]:
# ============================================================
# 4. TẢI DỮ LIỆU ĐÃ TIỀN XỬ LÝ (FAST-TRACK)
# ============================================================
BASE_URL = "https://raw.githubusercontent.com/Chinh-de/dss_ckan/main/backend/data/movie"

files_to_download = [
    ("ratings_final.npy", f"{DATA_DIR}/ratings_final.npy"),
    ("kg_final.npy", f"{DATA_DIR}/kg_final.npy"),
    ("item_index2entity_id.txt", f"{DATA_DIR}/item_index2entity_id.txt"),
    ("kg.txt", f"{DATA_DIR}/kg.txt")
]

def download_file(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f"  [Đã có] {dest} ({os.path.getsize(dest)/1e6:.2f} MB)")
        return True
    try:
        print(f"  [Tải về] {url} -> {dest}")
        urllib.request.urlretrieve(url, dest)
        print(f"  [Hoàn tất] {dest} ({os.path.getsize(dest)/1e6:.2f} MB)")
        return True
    except Exception as e:
        print(f"  [Lỗi tải] {e}")
        return False

print("Đang chuẩn bị dữ liệu...")
for fname, dest in files_to_download:
    download_file(f"{BASE_URL}/{fname}", dest)

# Nạp ma trận dữ liệu
rating_np = np.load(f"{DATA_DIR}/ratings_final.npy")
kg_np = np.load(f"{DATA_DIR}/kg_final.npy")

n_entity = int(max(np.max(kg_np[:, 0]), np.max(kg_np[:, 2]))) + 1
n_relation = int(np.max(kg_np[:, 1])) + 1
n_user = int(np.max(rating_np[:, 0])) + 1
n_item = int(np.max(rating_np[:, 1])) + 1

print(f"\n--- THỐNG KÊ DỮ LIỆU ---")
print(f"  - Số lượng User        : {n_user:,}")
print(f"  - Số lượng Item (Phim) : {n_item:,}")
print(f"  - Số bản ghi tương tác : {len(rating_np):,}")
print(f"  - Số Entity (KG)       : {n_entity:,}")
print(f"  - Số Relation (KG)     : {n_relation:,}")
print(f"  - Số bộ ba Tri thức    : {len(kg_np):,}")



## 5. Phân Chia Dữ Liệu (6:2:2) & Khởi Tạo Lan Truyền Tri Thức


In [ ]:
# ============================================================
# 5. CHIA TẬP TRAIN / EVAL / TEST (6:2:2) & KG PROPAGATION
# ============================================================
def collaboration_propagation(rating_np, train_indices):
    user_history_item_dict = dict()
    item_history_user_dict = dict()
    item_neighbor_item_dict = dict()

    for i in train_indices:
        user = rating_np[i][0]
        item = rating_np[i][1]
        rating = rating_np[i][2]
        if rating == 1:
            if user not in user_history_item_dict:
                user_history_item_dict[user] = []
            user_history_item_dict[user].append(item)
            if item not in item_history_user_dict:
                item_history_user_dict[item] = []
            item_history_user_dict[item].append(user)

    for item in item_history_user_dict.keys():
        item_neighbor_item = []
        for user in item_history_user_dict[item]:
            item_neighbor_item.extend(user_history_item_dict[user])
        item_neighbor_item_dict[item] = list(set(item_neighbor_item))

    item_list = set(rating_np[:, 1])
    for item in item_list:
        if item not in item_neighbor_item_dict:
            item_neighbor_item_dict[item] = [item]

    return user_history_item_dict, item_neighbor_item_dict

def dataset_split(rating_np):
    eval_ratio = 0.2
    test_ratio = 0.2
    n_ratings = rating_np.shape[0]

    np.random.seed(42)
    eval_indices = np.random.choice(n_ratings, size=int(n_ratings * eval_ratio), replace=False)
    left = set(range(n_ratings)) - set(eval_indices)
    test_indices = np.random.choice(list(left), size=int(n_ratings * test_ratio), replace=False)
    train_indices = list(left - set(test_indices))

    user_init_entity_set, item_init_entity_set = collaboration_propagation(rating_np, train_indices)

    train_indices = [i for i in train_indices if rating_np[i][0] in user_init_entity_set]
    eval_indices  = [i for i in eval_indices if rating_np[i][0] in user_init_entity_set]
    test_indices  = [i for i in test_indices if rating_np[i][0] in user_init_entity_set]

    train_data = rating_np[train_indices]
    eval_data  = rating_np[eval_indices]
    test_data  = rating_np[test_indices]
    return train_data, eval_data, test_data, user_init_entity_set, item_init_entity_set

def construct_kg(kg_np):
    kg = collections.defaultdict(list)
    for head, relation, tail in kg_np:
        kg[head].append((tail, relation))
    return kg

def kg_propagation(kg, init_entity_set, set_size, n_layer):
    # triple_sets: [n_obj][n_layer](h, r, t) x [set_size]
    triple_sets = collections.defaultdict(list)
    for obj, entities in init_entity_set.items():
        curr_entities = entities
        for l in range(n_layer):
            h, r, t = [], [], []
            for entity in curr_entities:
                for tail, relation in kg.get(entity, []):
                    h.append(entity)
                    t.append(tail)
                    r.append(relation)

            if len(h) == 0:
                if len(triple_sets[obj]) > 0:
                    triple_sets[obj].append(triple_sets[obj][-1])
                else:
                    triple_sets[obj].append(([obj] * set_size, [0] * set_size, [obj] * set_size))
            else:
                indices = np.random.choice(len(h), size=set_size, replace=(len(h) < set_size))
                h_sampled = [h[i] for i in indices]
                r_sampled = [r[i] for i in indices]
                t_sampled = [t[i] for i in indices]
                triple_sets[obj].append((h_sampled, r_sampled, t_sampled))
                curr_entities = t_sampled
    return triple_sets

print("Đang phân chia dữ liệu và xây dựng mạng lan truyền tri thức chuẩn tác giả...")
train_data, eval_data, test_data, user_init_entity_set, item_init_entity_set = dataset_split(rating_np)
kg = construct_kg(kg_np)
user_triple_set = kg_propagation(kg, user_init_entity_set, UTSS, N_LAYER)
item_triple_set = kg_propagation(kg, item_init_entity_set, ITSS, N_LAYER)

print(f"  - Train samples : {train_data.shape[0]:,}")
print(f"  - Eval samples  : {eval_data.shape[0]:,}")
print(f"  - Test samples  : {test_data.shape[0]:,}")
print("[OK] Dữ liệu chia đồng nhất sẵn sàng cho cả 2 mô hình!")


## 6. Định Nghĩa Cấu Trúc 2 Mô Hình
Định nghĩa độc lập:
- **Mô hình 1 (Baseline)**: Matrix Factorization (MF - Lọc cộng tác không dùng đồ thị tri thức).
- **Mô hình 2 (Đề xuất)**: CKAN (Collaborative Knowledge-aware Attentive Network - Có dùng đồ thị tri thức).


In [ ]:
# ============================================================
# 6. ĐỊNH NGHĨA CẤU TRÚC 2 MÔ HÌNH (MF vs CKAN)
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

# --- 6A. MÔ HÌNH BASELINE: MATRIX FACTORIZATION (MF - NO KG) ---
class MatrixFactorization(nn.Module):
    """
    Mô hình Biased Matrix Factorization (Lọc Cộng Tác thuần túy).
    Chỉ học từ ma trận tương tác User - Item mà KHÔNG dùng Đồ thị Tri thức.
    """
    def __init__(self, n_user, n_item, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_user, dim)
        self.item_emb = nn.Embedding(n_item, dim)
        self.user_bias = nn.Embedding(n_user, 1)
        self.item_bias = nn.Embedding(n_item, 1)
        
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        u_emb = self.user_emb(users)
        v_emb = self.item_emb(items)
        dot = (u_emb * v_emb).sum(dim=-1, keepdim=True)
        logits = dot + self.user_bias(users) + self.item_bias(items)
        return torch.sigmoid(logits.squeeze(-1))

    def get_all_scores(self, user_indices):
        u_emb = self.user_emb(user_indices)
        v_emb = self.item_emb.weight
        u_bias = self.user_bias(user_indices)
        v_bias = self.item_bias.weight.T
        logits = torch.matmul(u_emb, v_emb.T) + u_bias + v_bias
        return torch.sigmoid(logits)


# --- 6B. MÔ HÌNH ĐỀ XUẤT: CKAN (WITH KNOWLEDGE GRAPH) ---
class CKAN(nn.Module):
    """
    Mô hình Collaborative Knowledge-aware Attentive Network.
    Sử dụng mạng Attention để lan truyền sở thích qua các bước nhảy tri thức.
    """
    def __init__(self, n_entity, n_relation, dim, n_layer=1, agg="concat"):
        super().__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer
        self.agg = agg

        self.entity_emb = nn.Embedding(n_entity, dim)
        self.relation_emb = nn.Embedding(n_relation, dim)

        self.attention = nn.Sequential(
            nn.Linear(dim * 2, dim, bias=False), nn.ReLU(),
            nn.Linear(dim, dim, bias=False), nn.ReLU(),
            nn.Linear(dim, 1, bias=False), nn.Sigmoid()
        )
        self._init_weight()

    def _init_weight(self):
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        for m in self.attention:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def _knowledge_attention(self, h_emb, r_emb, t_emb):
        alpha = self.attention(torch.cat((h_emb, r_emb), dim=-1)).squeeze(-1)
        alpha = F.softmax(alpha, dim=-1)
        return torch.mul(alpha.unsqueeze(-1), t_emb).sum(dim=1)

    def get_user_embeddings(self, user_triple):
        user_embs = [self.entity_emb(user_triple[0][0]).mean(dim=1)]
        for l in range(self.n_layer):
            h = self.entity_emb(user_triple[0][l])
            r = self.relation_emb(user_triple[1][l])
            t = self.entity_emb(user_triple[2][l])
            user_embs.append(self._knowledge_attention(h, r, t))

        e_u = user_embs[0]
        if self.agg == "concat":
            for i in range(1, len(user_embs)):
                e_u = torch.cat((user_embs[i], e_u), dim=-1)
        elif self.agg == "sum":
            for i in range(1, len(user_embs)):
                e_u = e_u + user_embs[i]
        return e_u

    def get_item_embeddings(self, items, item_triple):
        item_embs = [self.entity_emb(items)]
        for l in range(self.n_layer):
            h = self.entity_emb(item_triple[0][l])
            r = self.relation_emb(item_triple[1][l])
            t = self.entity_emb(item_triple[2][l])
            item_embs.append(self._knowledge_attention(h, r, t))

        e_v = item_embs[0]
        if self.agg == "concat":
            for i in range(1, len(item_embs)):
                e_v = torch.cat((item_embs[i], e_v), dim=-1)
        elif self.agg == "sum":
            for i in range(1, len(item_embs)):
                e_v = e_v + item_embs[i]
        return e_v

    def forward(self, items, user_triple, item_triple):
        e_u = self.get_user_embeddings(user_triple)
        e_v = self.get_item_embeddings(items, item_triple)
        return torch.sigmoid((e_v * e_u).sum(dim=-1))

print("[OK] Đã định nghĩa xong 2 cấu trúc mô hình (MF và CKAN)!")



## 7. Huấn Luyện Mô Hình Baseline: Matrix Factorization (MF) Độc Lập
Huấn luyện mô hình cơ sở không dùng đồ thị tri thức để ghi nhận đường cong học tập và các chỉ số chuẩn.


In [ ]:
# ============================================================
# 7. HUẤN LUYỆN MATRIX FACTORIZATION (MF - NO KG)
# ============================================================
def evaluate_mf(model, data, b_size=BATCH_SIZE):
    model.eval()
    aucs, f1s = [], []
    with torch.no_grad():
        for start in range(0, data.shape[0], b_size):
            end = min(start + b_size, data.shape[0])
            batch = data[start:end]
            users = torch.LongTensor(batch[:, 0]).to(device)
            items = torch.LongTensor(batch[:, 1]).to(device)
            scores = model(users, items).cpu().numpy()
            labels = batch[:, 2]
            if len(np.unique(labels)) > 1:
                aucs.append(roc_auc_score(labels, scores))
                f1s.append(f1_score(labels, (scores >= 0.5).astype(int), zero_division=0))
    model.train()
    return float(np.mean(aucs)), float(np.mean(f1s))

# Khởi tạo mô hình MF
mf_model = MatrixFactorization(n_user, n_item, DIM).to(device)
opt_mf = torch.optim.Adam(mf_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
bce_loss = nn.BCELoss()

mf_history = {
    "epoch": [],
    "loss": [],
    "eval_auc": [],
    "eval_f1": [],
    "test_auc": [],
    "test_f1": []
}

print(f"=== BẮT ĐẦU HUẤN LUYỆN MATRIX FACTORIZATION ({N_EPOCH} EPOCHS) ===")
print("-" * 65)
print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Eval AUC':<10} {'Eval F1':<10} | {'Test AUC':<10} {'Test F1':<10}")
print("-" * 65)

for epoch in range(1, N_EPOCH + 1):
    np.random.shuffle(train_data)
    losses = []
    
    for start in range(0, train_data.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, train_data.shape[0])
        batch = train_data[start:end]
        users = torch.LongTensor(batch[:, 0]).to(device)
        items = torch.LongTensor(batch[:, 1]).to(device)
        labels = torch.FloatTensor(batch[:, 2]).to(device)

        opt_mf.zero_grad()
        preds = mf_model(users, items)
        loss = bce_loss(preds, labels)
        loss.backward()
        opt_mf.step()
        losses.append(loss.item())

    ev_auc, ev_f1 = evaluate_mf(mf_model, eval_data)
    te_auc, te_f1 = evaluate_mf(mf_model, test_data)

    mf_history["epoch"].append(epoch)
    mf_history["loss"].append(np.mean(losses))
    mf_history["eval_auc"].append(ev_auc)
    mf_history["eval_f1"].append(ev_f1)
    mf_history["test_auc"].append(te_auc)
    mf_history["test_f1"].append(te_f1)

    print(f"{epoch:<6} | {np.mean(losses):<12.4f} | {ev_auc:<10.4f} {ev_f1:<10.4f} | {te_auc:<10.4f} {te_f1:<10.4f}")

print("-" * 65)
print(f"[Hoàn tất] Huấn luyện MF Baseline! Peak Test AUC: {max(mf_history['test_auc']):.4f} | Peak Test F1: {max(mf_history['test_f1']):.4f}")



## 8. Huấn Luyện Mô Hình Đề Xuất: CKAN (With Knowledge Graph) Độc Lập
Huấn luyện mô hình CKAN với mạng Attention tri thức và tự động lưu file checkpoint tốt nhất (`ckan_model.pt`).


In [ ]:
# ============================================================
# 8. HUẤN LUYỆN CKAN (WITH KNOWLEDGE GRAPH)
# ============================================================
def to_triple_tensor(objs, triple_set, n_layer, dev):
    h, r, t = [], [], []
    for l in range(n_layer):
        h.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][0] for o in objs]).to(dev))
        r.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][1] for o in objs]).to(dev))
        t.append(torch.LongTensor([triple_set.get(o, triple_set[list(triple_set.keys())[0]])[l][2] for o in objs]).to(dev))
    return [h, r, t]

def evaluate_ckan(model, data, b_size=BATCH_SIZE):
    model.eval()
    aucs, f1s = [], []
    with torch.no_grad():
        for start in range(0, data.shape[0], b_size):
            end = min(start + b_size, data.shape[0])
            batch = data[start:end]
            items = torch.LongTensor(batch[:, 1]).to(device)
            u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, N_LAYER, device)
            i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, N_LAYER, device)
            scores = model(items, u_tr, i_tr).cpu().numpy()
            labels = batch[:, 2]
            if len(np.unique(labels)) > 1:
                aucs.append(roc_auc_score(labels, scores))
                f1s.append(f1_score(labels, (scores >= 0.5).astype(int), zero_division=0))
    model.train()
    return float(np.mean(aucs)), float(np.mean(f1s))

# Khởi tạo mô hình CKAN
ckan_model = CKAN(n_entity, n_relation, DIM, N_LAYER, AGG).to(device)
opt_ckan = torch.optim.Adam(ckan_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)

ckan_history = {
    "epoch": [],
    "loss": [],
    "eval_auc": [],
    "eval_f1": [],
    "test_auc": [],
    "test_f1": []
}

best_ckan_auc = 0.0

print(f"=== BẮT ĐẦU HUẤN LUYỆN CKAN WITH KNOWLEDGE GRAPH ({N_EPOCH} EPOCHS) ===")
print("-" * 65)
print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Eval AUC':<10} {'Eval F1':<10} | {'Test AUC':<10} {'Test F1':<10}")
print("-" * 65)

for epoch in range(1, N_EPOCH + 1):
    np.random.shuffle(train_data)
    losses = []

    for start in range(0, train_data.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, train_data.shape[0])
        batch = train_data[start:end]
        items = torch.LongTensor(batch[:, 1]).to(device)
        labels = torch.FloatTensor(batch[:, 2]).to(device)
        u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, N_LAYER, device)
        i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, N_LAYER, device)

        opt_ckan.zero_grad()
        preds = ckan_model(items, u_tr, i_tr)
        loss = bce_loss(preds, labels)
        loss.backward()
        opt_ckan.step()
        losses.append(loss.item())

    ev_auc, ev_f1 = evaluate_ckan(ckan_model, eval_data)
    te_auc, te_f1 = evaluate_ckan(ckan_model, test_data)

    ckan_history["epoch"].append(epoch)
    ckan_history["loss"].append(np.mean(losses))
    ckan_history["eval_auc"].append(ev_auc)
    ckan_history["eval_f1"].append(ev_f1)
    ckan_history["test_auc"].append(te_auc)
    ckan_history["test_f1"].append(te_f1)

    print(f"{epoch:<6} | {np.mean(losses):<12.4f} | {ev_auc:<10.4f} {ev_f1:<10.4f} | {te_auc:<10.4f} {te_f1:<10.4f}")

    # Đóng gói và lưu checkpoint tốt nhất
    if te_auc > best_ckan_auc:
        best_ckan_auc = te_auc
        checkpoint_bundle = {
            "model_state_dict": ckan_model.state_dict(),
            "args": {
                "dim": DIM,
                "n_layer": N_LAYER,
                "agg": AGG,
                "batch_size": BATCH_SIZE,
                "use_cuda": torch.cuda.is_available()
            },
            "n_entity": n_entity,
            "n_relation": n_relation,
            "best_auc": best_ckan_auc,
            "best_f1": te_f1
        }
        torch.save(checkpoint_bundle, "./models/ckan_model.pt")

print("-" * 65)
print(f"[Hoàn tất] Huấn luyện CKAN! Peak Test AUC: {best_ckan_auc:.4f}")
print("Đã tự động lưu checkpoint xuất sắc nhất vào: ./models/ckan_model.pt")



## 9. Đánh Giá Top-K Recommendation (All-Ranking Protocol)
Đánh giá so sánh trực tiếp khả năng gợi ý thực tế giữa **Matrix Factorization (MF)** và **CKAN (With KG)**:
- Giao thức **All-ranking**: Chấm điểm toàn bộ các phim chưa tương tác trong kho của người dùng.
- Thang đo **Recall@K**: Tỷ lệ tìm trúng các phim người dùng yêu thích trong tập test.
- Thang đo **NDCG@K**: Đánh giá vị trí hiển thị của các phim hit trong Top-K (càng ở top đầu điểm càng cao).


In [ ]:
# ============================================================
# 9. TOP-K RECOMMENDATION EVALUATION (MF vs CKAN)
# ============================================================
from tqdm import tqdm

def dcg_at_k(r, k):
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.0

def ndcg_at_k(r, k, ground_truth_count):
    dcg = dcg_at_k(r, k)
    ideal_r = [1] * min(k, ground_truth_count)
    idcg = dcg_at_k(ideal_r, k)
    return dcg / idcg if idcg > 0 else 0.0

def topk_eval_all(mf_model, ckan_model, train_data, test_data, user_triple_set, item_triple_set, n_layer, n_item, device, k_list=[5, 10, 20]):
    mf_model.eval()
    ckan_model.eval()

    user_train_pos = defaultdict(set)
    for u, i, r in train_data:
        if r == 1:
            user_train_pos[u].add(i)

    user_test_pos = defaultdict(set)
    for u, i, r in test_data:
        if r == 1:
            user_test_pos[u].add(i)

    # Đánh giá trên toàn bộ người dùng có tương tác test (chuẩn 100% benchmark, không lấy mẫu ngẫu nhiên)
    eval_users = sorted(list(set(user_train_pos.keys()) & set(user_test_pos.keys())))
    all_items = np.arange(n_item)

    print(f"Đang tính ma trận Top-K ranking cho toàn bộ {n_item:,} phim và {len(eval_users):,} người dùng test...")

    with torch.no_grad():
        # --- 1. ĐÁNH GIÁ MF (GPU Vectorized) ---
        mf_u_tens = torch.LongTensor(eval_users).to(device)
        mf_all_scores = mf_model.get_all_scores(mf_u_tens) # [len(eval_users), n_item]

        # --- 2. ĐÁNH GIÁ CKAN (GPU Vectorized) ---
        item_tensor = torch.LongTensor(all_items).to(device)
        all_i_tr = to_triple_tensor(all_items.tolist(), item_triple_set, n_layer, device)
        E_V = ckan_model.get_item_embeddings(item_tensor, all_i_tr) # [n_item, 128]

        all_u_tr = to_triple_tensor(eval_users, user_triple_set, n_layer, device)
        E_U = ckan_model.get_user_embeddings(all_u_tr) # [len(eval_users), 128]
        ckan_all_scores = torch.sigmoid(torch.matmul(E_U, E_V.T)) # [len(eval_users), n_item]

        # Gán điểm âm vô cực cho các phim đã xem trong tập train
        for idx, u in enumerate(eval_users):
            seen = list(user_train_pos[u])
            if len(seen) > 0:
                mf_all_scores[idx, seen] = -1e9
                ckan_all_scores[idx, seen] = -1e9

        mf_top_indices = torch.topk(mf_all_scores, k=max(k_list), dim=-1).indices.cpu().numpy()
        ckan_top_indices = torch.topk(ckan_all_scores, k=max(k_list), dim=-1).indices.cpu().numpy()

    mf_recalls, mf_ndcgs = {k: [] for k in k_list}, {k: [] for k in k_list}
    ckan_recalls, ckan_ndcgs = {k: [] for k in k_list}, {k: [] for k in k_list}

    print(f"Đang tổng hợp so sánh trên toàn bộ {len(eval_users):,} người dùng...")
    for idx, u in enumerate(eval_users):
        ground_truth = user_test_pos[u]
        mf_user_top = mf_top_indices[idx]
        ckan_user_top = ckan_top_indices[idx]

        for k in k_list:
            # MF
            mf_hits = [1 if item in ground_truth else 0 for item in mf_user_top[:k]]
            mf_recalls[k].append(sum(mf_hits) / len(ground_truth))
            mf_ndcgs[k].append(ndcg_at_k(mf_hits, k, len(ground_truth)))

            # CKAN
            ckan_hits = [1 if item in ground_truth else 0 for item in ckan_user_top[:k]]
            ckan_recalls[k].append(sum(ckan_hits) / len(ground_truth))
            ckan_ndcgs[k].append(ndcg_at_k(ckan_hits, k, len(ground_truth)))

    return {
        "MF": {
            "Recall": {k: float(np.mean(mf_recalls[k])) for k in k_list},
            "NDCG": {k: float(np.mean(mf_ndcgs[k])) for k in k_list}
        },
        "CKAN": {
            "Recall": {k: float(np.mean(ckan_recalls[k])) for k in k_list},
            "NDCG": {k: float(np.mean(ckan_ndcgs[k])) for k in k_list}
        }
    }

topk_cmp = topk_eval_all(mf_model, ckan_model, train_data, test_data, user_triple_set, item_triple_set, N_LAYER, n_item, device, k_list=[5, 10, 20])

print("-" * 75)
print(f"{'K':<5} | {'MF Recall@K':<15} {'CKAN Recall@K':<15} | {'MF NDCG@K':<15} {'CKAN NDCG@K':<15}")
print("-" * 75)
for k in [5, 10, 20]:
    print(f"{k:<5} | {topk_cmp['MF']['Recall'][k]:<15.4f} {topk_cmp['CKAN']['Recall'][k]:<15.4f} | {topk_cmp['MF']['NDCG'][k]:<15.4f} {topk_cmp['CKAN']['NDCG'][k]:<15.4f}")
print("-" * 75)



## 10. Trực Quan Hóa So Sánh Toàn Diện (CTR Prediction & Top-K Recommendation)
Đặt kết quả đánh giá của cả 2 mô hình lên 4 biểu đồ trực quan.


In [ ]:
# ============================================================
# 10. TRỰC QUAN HÓA SO SÁNH HIỆU NĂNG
# ============================================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

epochs = mf_history["epoch"]

# 1. So sánh Test ROC-AUC
axes[0, 0].plot(epochs, ckan_history["test_auc"], label="CKAN (With KG)", color="#E5A93C", linewidth=2.5, marker="o")
axes[0, 0].plot(epochs, mf_history["test_auc"], label="Matrix Factorization (MF)", color="#3B82F6", linewidth=2, linestyle="--", marker="s")
axes[0, 0].set_title("CTR Prediction: Test ROC-AUC", fontsize=12, fontweight="bold")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("ROC-AUC")
axes[0, 0].legend(loc="lower right")

# 2. So sánh Test F1-Score
axes[0, 1].plot(epochs, ckan_history["test_f1"], label="CKAN (With KG)", color="#E5A93C", linewidth=2.5, marker="o")
axes[0, 1].plot(epochs, mf_history["test_f1"], label="Matrix Factorization (MF)", color="#3B82F6", linewidth=2, linestyle="--", marker="s")
axes[0, 1].set_title("CTR Prediction: Test F1-Score", fontsize=12, fontweight="bold")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("F1-Score")
axes[0, 1].legend(loc="lower right")

# 3. So sánh Top-K Recall@K
k_vals = [5, 10, 20]
x = np.arange(len(k_vals))
width = 0.35

mf_r = [topk_cmp["MF"]["Recall"][k] for k in k_vals]
ckan_r = [topk_cmp["CKAN"]["Recall"][k] for k in k_vals]

axes[1, 0].bar(x - width/2, mf_r, width, label='MF (No KG)', color='#93c5fd')
axes[1, 0].bar(x + width/2, ckan_r, width, label='CKAN (With KG)', color='#E5A93C')
axes[1, 0].set_title("Top-K Recommendation: Recall@K", fontsize=12, fontweight="bold")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels([f"K={k}" for k in k_vals])
axes[1, 0].set_ylabel("Recall")
axes[1, 0].legend()

# 4. So sánh Top-K NDCG@K
mf_n = [topk_cmp["MF"]["NDCG"][k] for k in k_vals]
ckan_n = [topk_cmp["CKAN"]["NDCG"][k] for k in k_vals]

axes[1, 1].bar(x - width/2, mf_n, width, label='MF (No KG)', color='#93c5fd')
axes[1, 1].bar(x + width/2, ckan_n, width, label='CKAN (With KG)', color='#E5A93C')
axes[1, 1].set_title("Top-K Recommendation: NDCG@K", fontsize=12, fontweight="bold")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels([f"K={k}" for k in k_vals])
axes[1, 1].set_ylabel("NDCG")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# In bảng tổng kết
print("\n--- BẢNG TỔNG KẾT TẤT CẢ THANG ĐO (MF vs CKAN) ---")
print("1. CTR PREDICTION:")
print(f"   • Matrix Factorization (MF) Test AUC : {max(mf_history['test_auc']):.4f} | F1: {max(mf_history['test_f1']):.4f}")
print(f"   • CKAN (With Knowledge Graph) Test AUC: {max(ckan_history['test_auc']):.4f} | F1: {max(ckan_history['test_f1']):.4f}")
print("2. TOP-K RECOMMENDATION:")
for k in [5, 10, 20]:
    r_gain = (topk_cmp['CKAN']['Recall'][k] - topk_cmp['MF']['Recall'][k]) * 100
    n_gain = (topk_cmp['CKAN']['NDCG'][k] - topk_cmp['MF']['NDCG'][k]) * 100
    print(f"   • K={k:2d}: Recall@{k}: MF={topk_cmp['MF']['Recall'][k]:.4f} vs CKAN={topk_cmp['CKAN']['Recall'][k]:.4f} (CKAN tăng +{r_gain:.2f}%)")
    print(f"           NDCG@{k}  : MF={topk_cmp['MF']['NDCG'][k]:.4f} vs CKAN={topk_cmp['CKAN']['NDCG'][k]:.4f} (CKAN tăng +{n_gain:.2f}%)")



## 11. Tải Mô Hình CKAN Đã Huấn Luyện Về Máy Tính
Chạy ô code dưới đây để kích hoạt trình duyệt tự động tải file `ckan_model.pt` về máy:


In [ ]:
# ============================================================
# 11. TẢI FILE MÔ HÌNH VỀ MÁY TÍNH
# ============================================================
from google.colab import files

MODEL_PATH = "./models/ckan_model.pt"

if os.path.exists(MODEL_PATH):
    file_size_mb = os.path.getsize(MODEL_PATH) / 1e6
    print(f"Đang tải file mô hình {MODEL_PATH} ({file_size_mb:.2f} MB) về máy tính...")
    files.download(MODEL_PATH)
    print("[OK] Đã kích hoạt tải về trên trình duyệt!")
else:
    print("[Lỗi] Không tìm thấy file checkpoint. Vui lòng chạy bước huấn luyện CKAN ở mục 8 trước.")



### Hướng Dẫn Sử Dụng Mô Hình Trong Dự Án Local
1. Đặt file `ckan_model.pt` vừa tải về vào thư mục backend của dự án:
   ```bash
   dss_ckan_movie_recommender_system/backend/models/ckan_model.pt
   ```
2. Khởi động lại Backend FastAPI:
   ```bash
   uv run uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload
   ```
3. Backend sẽ tự động phát hiện và nạp trọng số mô hình:
   ```log
   INFO: Loading CKAN checkpoint from backend/models/ckan_model.pt...
   INFO: CKAN checkpoint loaded successfully.
   ```
Toàn bộ hệ thống gợi ý và đồ thị tri thức trên frontend `http://localhost:5173` sẽ lập tức sử dụng mô hình vừa huấn luyện!
